# Stage 4 — Official Experiments

Runs all four methods on Mistral-7B with WikiText-103 and LongBench.
Uses `run_benchmark_official()` which evaluates perplexity autoregressively under each method's own cache strategy.

| Method | Description |
|--------|-------------|
| `full_cache` | Standard HuggingFace decoding, no eviction |
| `sliding_window` | Keep only most recent 256 tokens |
| `naive_truncation` | Hard-cut prompt to 256 tokens before generation |
| `adaptive` | Our method: importance scoring + INT8 compression + budget-triggered eviction |

**Datasets:**
- WikiText-103: standard perplexity benchmark (2048 token context)
- LongBench (qasper): long-context evaluation, avg 3600 words (4096 token context), used in H2O and SnapKV

## 1. Setup

In [ ]:
# Clone repo and install dependencies (run once per Colab session)
!git clone https://github.com/yanghao13111/adaptive-kv-cache.git 2>/dev/null || echo "Repo already cloned"
%cd adaptive-kv-cache
!git pull
# Downgrade datasets to support LongBench (datasets 4.0.0 dropped script support)
!pip install datasets==2.19.0 transformers accelerate -q


In [ ]:
# HuggingFace login — required to download Mistral-7B
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
import sys
sys.path.insert(0, '/content/adaptive-kv-cache')

from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'mistralai/Mistral-7B-v0.1'
print(f'Device: {DEVICE}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    attn_implementation='eager',
)
model.eval()
print(f'Model loaded: {MODEL_NAME}')
print(f'Layers: {model.config.num_hidden_layers}')

## 2. Experiment Configuration

All four methods share the same base config.
Method-specific parameters are in `method_kwargs`.

In [ ]:
from src.eval.benchmark import run_benchmark_official, save_results

BASE_CONFIG = {
    'model': MODEL_NAME,
    'dataset': 'wikitext-103',
    'num_samples': 50,
    'max_new_tokens': 256,
    'context_len': 2048,
}

CONFIGS = [
    {**BASE_CONFIG, 'method': 'full_cache'},
    {**BASE_CONFIG, 'method': 'sliding_window',
     'method_kwargs': {'window_size': 256}},
    {**BASE_CONFIG, 'method': 'naive_truncation',
     'method_kwargs': {'max_cache_size': 256}},
    {**BASE_CONFIG, 'method': 'adaptive',
     'method_kwargs': {
         'memory_budget_gb': 4.0,
         'recent_window': 256,
         'compress_dtype': 'int8',
         'sink_tokens': 4,
         'score_decay': 0.9,
     }},
]

print('Configs ready — 4 methods x WikiText-103')

## 3. WikiText-103 Results

Run all four methods on WikiText-103 (50 samples, 2048 token context).
This is the standard perplexity benchmark used in H2O, SnapKV, and StreamingLLM.

In [ ]:
import pandas as pd

wikitext_results = []
for config in CONFIGS:
    print(f"\n{'='*50}")
    print(f"Running: {config['method']} on WikiText-103")
    print(f"{'='*50}")
    result = run_benchmark_official(config, model, tokenizer)
    save_results(result)
    wikitext_results.append(result)
    print(f"  perplexity: {result['perplexity']}")
    print(f"  latency:    {result['avg_latency_ms_per_token']} ms/token")
    print(f"  memory:     {result['avg_peak_memory_gb']} GB")

df_wiki = pd.DataFrame(wikitext_results)[[
    'method', 'avg_latency_ms_per_token',
    'avg_throughput_tokens_per_sec', 'avg_peak_memory_gb', 'perplexity'
]]
print('\n=== WikiText-103 Results ===')
display(df_wiki)

## 4. LongBench Results (context_len=4096)

LongBench (qasper subset) contains scientific papers with long contexts (avg 3600 words, max 14640 words).
This is where KV cache pressure is highest — our method's memory savings should be most visible here.
LongBench is used in H2O and SnapKV for long-context evaluation, so results are directly comparable.

In [ ]:
LONGBENCH_CONFIGS = [
    {**c, 'dataset': 'longbench', 'num_samples': 20, 'context_len': 4096}
    for c in CONFIGS
]

longbench_results = []
for config in LONGBENCH_CONFIGS:
    print(f"\n{'='*50}")
    print(f"Running: {config['method']} on LongBench")
    print(f"{'='*50}")
    result = run_benchmark_official(config, model, tokenizer)
    save_results(result)
    longbench_results.append(result)
    print(f"  perplexity: {result['perplexity']}")
    print(f"  memory:     {result['avg_peak_memory_gb']} GB")

df_longbench = pd.DataFrame(longbench_results)[[
    'method', 'avg_latency_ms_per_token',
    'avg_throughput_tokens_per_sec', 'avg_peak_memory_gb', 'perplexity'
]]
print('\n=== LongBench Results (context_len=4096) ===')
display(df_longbench)

## 5. Combined Results Summary

Load all saved CSVs and display a combined table across both datasets.

In [ ]:
import glob
import pandas as pd

csv_files = glob.glob('experiments/results/*.csv')
if csv_files:
    df_all = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    df_all = df_all.sort_values(['dataset', 'method']).reset_index(drop=True)
    display(df_all)
    df_all.to_csv('experiments/results/combined_results.csv', index=False)
    print('Saved to experiments/results/combined_results.csv')
else:
    print('No results yet — run sections 3 and 4 first.')